# Trip-Purpose Constraint Function

## Overview

This notebook defines and validates a reusable demographic-based trip-purpose constraint function.

Profiles with `Industry = Does not apply` cannot have `Commuting` or `Business` as trip purposes. Employed profiles who work mainly at or from home cannot have `Commuting`, but retain `Business`.

After infeasible trip purposes are removed, the remaining probabilities are renormalised to sum to one.

Profiles from the demographic profile pool are used as validation cases only and do not represent population sampling.

## 1. Import Libraries

In [1]:
import json

from copy import deepcopy
from pathlib import Path

import pandas as pd

## 2. Set Up File Paths

In [2]:
project_root = Path.cwd().parents[1]

population_file = (
    project_root
    / "output"
    / "demographic_profiles"
    / "constrained_representative_population.parquet"
)

behavioural_evidence_file = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence.json"
)

## 3. Load Input Data

In [3]:
population = pd.read_parquet(population_file)

with open(
    behavioural_evidence_file,
    "r",
    encoding="utf-8",
) as file:
    behavioural_evidence = json.load(file)

print("Population loaded:", population.shape)
print("Trip purposes loaded:", len(behavioural_evidence))

Population loaded: (2386560, 14)
Trip purposes loaded: 14


## 4. Define Trip-Purpose Constraint Function

## Trip-Purpose Constraint

The function accepts one representative profile and its trip-purpose probability distribution.

- If `Industry = Does not apply`, `Commuting` and `Business` are removed.
- If the profile has an industry but works mainly at or from home, only `Commuting` is removed.
- Other profiles retain the original trip-purpose distribution.

The remaining probabilities are renormalised after a constraint is applied.

In [4]:
def apply_trip_purpose_constraint(
    profile,
    trip_purpose_probabilities,
):
    constrained = deepcopy(
        trip_purpose_probabilities
    )

    if profile["Industry"] == "Does not apply":

        constrained.pop("Commuting", None)
        constrained.pop("Business", None)

    elif (
        profile["Distance travelled to work"]
        == "Works mainly from home"
    ):

        constrained.pop("Commuting", None)

    else:

        return constrained

    total = sum(constrained.values())

    return {
        purpose: probability / total
        for purpose, probability
        in constrained.items()
    }

## 5. Select Test Profiles

## Test Profiles

Three feasible profiles are selected from the demographic profile pool:

- a profile with `Industry = Does not apply`;
- an employed profile who works mainly at or from home;
- a workplace-based worker used as the control case.

In [5]:
WFH_LABEL = "Works mainly from home"

profile_columns = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

industry_candidates = population[
    population["Industry"] == "Does not apply"
]

wfh_candidates = population[
    (population["Industry"] != "Does not apply")
    & (
        population["Distance travelled to work"]
        == WFH_LABEL
    )
]

workplace_candidates = population[
    (population["Industry"] != "Does not apply")
    & (
        ~population[
            "Distance travelled to work"
        ].isin(
            {
                "Does not apply",
                WFH_LABEL,
            }
        )
    )
]

if industry_candidates.empty:
    raise ValueError(
        "No Industry = Does not apply profile found."
    )

if wfh_candidates.empty:
    raise ValueError(
        "No work-from-home profile found."
    )

if workplace_candidates.empty:
    raise ValueError(
        "No workplace-based profile found."
    )

test_profiles = {
    "Industry does not apply": (
        industry_candidates.iloc[0][
            profile_columns
        ].to_dict()
    ),
    "Works mainly from home": (
        wfh_candidates.iloc[0][
            profile_columns
        ].to_dict()
    ),
    "Workplace-based worker": (
        workplace_candidates.iloc[0][
            profile_columns
        ].to_dict()
    ),
}

test_profiles_df = pd.DataFrame(
    test_profiles
).T

test_profiles_df.index.name = "Profile Type"

display(
    test_profiles_df.reset_index()[
        [
            "Profile Type",
            "Age",
            "Gender",
            "Economic Activity Status",
            "Industry",
            "Distance travelled to work",
        ]
    ]
)

,Profile Type,Age,Gender,Economic Activity Status,Industry,Distance travelled to work
0,Industry does not apply,0-4,Female,Does not apply,Does not apply,Does not apply
1,Works mainly from home,12-17,Female,Economically active and a full-time student: I...,J Information and communication,Works mainly from home
2,Workplace-based worker,12-17,Female,Economically active and a full-time student: I...,J Information and communication,10km to less than 30km


## 6. Test and Validate Function

The function is applied to each selected profile.

Validation compares the original and constrained trip-purpose distributions and reports:

- the number of remaining trip purposes;
- which trip purposes were removed; and
- whether the remaining probabilities sum to one.

In [6]:
validation_results = []

for profile_type, profile in test_profiles.items():

    original_probabilities = {}

    for purpose, evidence in behavioural_evidence.items():

        matching_records = [
            record
            for record in evidence["baseline_profile"]
            if record["Age"] == profile["Age"]
            and record["Gender"] == profile["Gender"]
        ]

        original_probabilities[purpose] = (
            matching_records[0][
                "Trip Purpose Probability"
            ]
        )

    constrained_probabilities = (
        apply_trip_purpose_constraint(
            profile,
            original_probabilities,
        )
    )

    removed_purposes = [
        purpose
        for purpose in original_probabilities
        if purpose not in constrained_probabilities
    ]

    if profile["Industry"] == "Does not apply":

        expected_removed = [
            "Commuting",
            "Business",
        ]

    elif (
        profile["Distance travelled to work"]
        == WFH_LABEL
    ):

        expected_removed = [
            "Commuting",
        ]

    else:

        expected_removed = []

    probability_sum = sum(
        constrained_probabilities.values()
    )

    purposes_correct = (
        sorted(removed_purposes)
        == sorted(expected_removed)
    )

    probabilities_correct = (
        abs(probability_sum - 1) < 1e-8
    )

    if purposes_correct and probabilities_correct:
        result = "PASS"
    else:
        result = "FAIL"

    validation_results.append(
        {
            "Profile Type": profile_type,
            "Original Purpose Count": len(
                original_probabilities
            ),
            "Constrained Purpose Count": len(
                constrained_probabilities
            ),
            "Removed Trip Purposes": (
                ", ".join(removed_purposes)
                if removed_purposes
                else "None"
            ),
            "Probability Sum": probability_sum,
            "Validation": result,
        }
    )

validation_results = pd.DataFrame(
    validation_results
)

display(validation_results)

if (
    validation_results["Validation"]
    == "PASS"
).all():
    print("Trip-purpose constraint validation: PASS")
else:
    print("Trip-purpose constraint validation: FAIL")

,Profile Type,Original Purpose Count,Constrained Purpose Count,Removed Trip Purposes,Probability Sum,Validation
0,Industry does not apply,14,12,"Commuting, Business",1.0,PASS
1,Works mainly from home,14,13,Commuting,1.0,PASS
2,Workplace-based worker,14,14,None,1.0,PASS


Trip-purpose constraint validation: PASS
